# Session 13: Regression: explaining and predicting a number

Model weekly orders for the Lakeside Mart branches. Fit it with numpy, read each coefficient in a sentence, then see the same model in statsmodels.

**SOLUTION VERSION. Tutor only.** Run the cells from the top. Each cell that starts with '# Your turn' is yours to complete. Run it, then run the check cell after it: the check tells you if you are right and what to fix.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, zipfile, urllib.request
DATA_ZIP_URL = ""   # your tutor may give you a link; leave empty to upload data.zip instead
if not os.path.exists("orders.csv"):
    if DATA_ZIP_URL:
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

## Task 1. Load the weeks
Read `branch_weeks.csv` into `bw`. Print its shape and store the correlation between `orders` and `footfall` in `r`.

In [ ]:
import pandas as pd
import numpy as np

bw = pd.read_csv('branch_weeks.csv')
print(bw.shape)
r = bw.orders.corr(bw.footfall)
print(round(r, 2))

In [ ]:
assert bw.shape == (832, 7), 'Check the file name'
assert 0.80 <= r <= 0.88, 'Use bw.orders.corr(bw.footfall)'
print('Task 1 correct')

## Task 2. One predictor
Fit a straight line of `orders` on `footfall` with `np.polyfit`. Keep the results in `slope` and `intercept`.

In [ ]:
import numpy as np

slope, intercept = np.polyfit(bw.footfall, bw.orders, 1)
print(round(slope, 4), round(intercept, 2))

In [ ]:
assert 0.017 <= slope <= 0.020, 'The slope should be near 0.0185'
assert 0.5 <= intercept <= 2.0, 'The intercept should be near 1.2'
print('Task 2 correct')

## Task 3. Three predictors
Build `X` with a column of ones, then footfall, promo_on and rainfall_mm. Put the orders in `y`. Solve with `np.linalg.lstsq` into `coef`. The answer key says the footfall coefficient is 0.0188, promotion 6.80 and rain -0.118.

In [ ]:
X = np.column_stack([np.ones(len(bw)), bw.footfall, bw.promo_on, bw.rainfall_mm])
y = bw.orders.values
coef, *_ = np.linalg.lstsq(X, y, rcond=None)
print(coef.round(4))

In [ ]:
assert abs(coef[1] - 0.0188) < 0.0005, 'Footfall should be near 0.0188'
assert abs(coef[2] - 6.80) < 0.3, 'Promotion should be near 6.8'
assert abs(coef[3] + 0.118) < 0.01, 'Rain should be near -0.118'
print('Task 3 correct')

## Task 4. R squared, the typical miss and a residual plot
Compute the residuals `miss` (actual minus fitted), then `r2` as 1 minus the variance of the misses divided by the variance of `y`. Compute `typical`, the standard deviation of the misses. Then plot the misses against the fitted orders and look for a bend.

In [ ]:
miss = y - X @ coef
r2 = 1 - miss.var() / y.var()
typical = miss.std()
print(round(r2, 3), round(typical, 2))

import matplotlib.pyplot as plt
plt.scatter(X @ coef, miss, s=8)
plt.axhline(0, color='red')
plt.xlabel('Fitted orders')
plt.ylabel('Miss')
plt.show()

In [ ]:
assert 0.78 <= r2 <= 0.81, 'R squared should be near 0.795'
assert 5.5 <= typical <= 6.6, 'The typical miss should be near 6 orders'
print('Task 4 correct')

## Task 5. Predict one week
A branch expects 2,000 people, a promotion and 20 mm of rain. Compute `pred`, the model's orders for that week.

In [ ]:
week = np.array([1, 2000, 1, 20])
pred = week @ coef
print(round(pred, 1))

In [ ]:
assert 43 <= pred <= 46, 'The prediction should be near 44'
print('Task 5 correct')

## Task 6. Are rain and footfall linked?
Compute `link`, the correlation of `footfall` with `rainfall_mm`. Then fit orders on rain alone and keep its slope in `rain_slope`. Compare it with the rain coefficient in `coef`.

In [ ]:
link = bw.footfall.corr(bw.rainfall_mm)
Xr = np.column_stack([np.ones(len(bw)), bw.rainfall_mm])
rain_only, *_ = np.linalg.lstsq(Xr, y, rcond=None)
rain_slope = rain_only[1]
print(round(link, 3), round(rain_slope, 4), round(coef[3], 4))

In [ ]:
assert abs(link) < 0.05, 'The link should be close to zero'
assert abs(rain_slope - coef[3]) < 0.02, 'The two rain numbers should be close'
print('Task 6 correct')

## Task 7. The branch as a category
Turn `branch_id` into 0 and 1 columns with `pd.get_dummies(..., drop_first=True, dtype=float)`. Add them to `X`, refit, and keep the new R squared in `r2b`.

In [ ]:
dummies = pd.get_dummies(bw.branch_id, drop_first=True, dtype=float)
Xb = np.column_stack([X, dummies.values])
cb, *_ = np.linalg.lstsq(Xb, y, rcond=None)
r2b = 1 - (y - Xb @ cb).var() / y.var()
print(round(r2b, 3))

In [ ]:
assert 0.79 <= r2b <= 0.81, 'R squared should barely change'
assert r2b >= r2 - 1e-9, 'Adding columns cannot lower R squared'
print('Task 7 correct')

## Task 8. The same model in statsmodels
Fit the formula version into `model`, with orders explained by footfall, promo_on and rainfall_mm. Colab has statsmodels. The coefficients should match your numpy answer.

In [ ]:
import statsmodels.formula.api as smf

model = smf.ols('orders ~ footfall + promo_on + rainfall_mm', data=bw).fit()
print(model.params.round(4))
print(round(model.rsquared, 3))

In [ ]:
assert abs(model.params['footfall'] - coef[1]) < 0.001, 'Footfall should match numpy'
assert 0.78 <= model.rsquared <= 0.81, 'R squared should be near 0.795'
print('Task 8 correct')

## Task 9. Why is weekly sales noisier than orders?
Fit the same three predictors to `sales_ugx` instead of orders. Keep the R squared in `r2_sales` and compare it with `r2`.

In [ ]:
t = bw.sales_ugx.values
cs, *_ = np.linalg.lstsq(X, t, rcond=None)
r2_sales = 1 - (t - X @ cs).var() / t.var()
print(round(r2, 3), round(r2_sales, 3))

In [ ]:
assert 0.30 <= r2_sales <= 0.42, 'R squared for sales should be near 0.357'
assert r2_sales < r2 - 0.2, 'Sales should be much noisier than orders'
print('Task 9 correct')

## Task 10. Write it up and write your capstone question
On a page, write: your question, one plain sentence for each coefficient (footfall per 1,000 people, promotion, rain per 10 mm), R squared in words, and one limit. Then check the footfall slope in Sheets with `=SLOPE(orders_range, footfall_range)` and `=RSQ(orders_range, footfall_range)`. Finally write your capstone question in one sentence, and name the column you want to explain or predict. You choose the dataset in Session 14.